# Verifying measurement-assisted uncomputation

Adapted from Qualtran v0.7.0 `simulation/MBUC.ipynb`, commit `bcb17bb25360ed68ce3bc084645db3e929799389` (Google LLC, Apache-2.0). [Locked source](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/simulation/MBUC.ipynb).

Reversing a computation can be costly. Measuring a computed bit in X instead produces an outcome-dependent phase; feedback must remove that relative phase. See Gidney's [2017 addition paper](https://arxiv.org/abs/1709.06648) and [2019 verification article](https://algassert.com/post/1903).

Run from the repository root. This notebook composes existing native SSA, measurement, phase and exact-AND-adjoint owners. The reference notebook contains stale-variable checks described below; native checks use the current state.

## Checking the AND adjoint

The numerical guard accepts erasure only when target stores q1 AND q2. Controls are retained and target has no logical output. This is a classical validity check, not a quantum run-time error handler.

In [ ]:
import itertools
import numpy as np
from examples.algorithms.measurement_uncomputation import (
    checked_and_adjoint, uncomputation_program, uncomputation_state, seeded_trials, EraseAnd, draw_erasure_steps,
)
for q1, q2 in itertools.product(range(2), repeat=2):
    target = q1 & q2
    result = checked_and_adjoint(q1, q2, target)
    np.testing.assert_array_equal(result["ctrl"], [q1, q2])
    print(q1, q2, target, "passed")

An inconsistent target raises ValueError in the numerical model. Error wording is native; the reference guard and its failure domain are tested separately.

In [ ]:
try:
    checked_and_adjoint(1, 1, 0)
except ValueError as error:
    print(error)

## The naive attempt

Measure target in X and discard its classical record. q1 and q2 are retained. The numerical graph allocates no scratch; its record is owned until discard. The physical demonstration additionally resets the caller-owned consumed target. The native ordered instruction table replaces the source logical musical score; their abstraction levels and layout differ.

In [ ]:
from flagquantum.dynamic import DynamicCircuit
throw_out_target = uncomputation_program(feedback=False)
naive_circuit = DynamicCircuit(3)
naive_circuit.append(EraseAnd(feedback=False))
naive_figure = draw_erasure_steps(naive_circuit)

## Seeded fuzz testing

The existing phased numerical state binds sampled outcomes using an explicit NumPy generator. The seed 123 and ten trials match the reference. This is a basis-state phase model, not a source of quantum branch probabilities.

In [ ]:
naive_phases = seeded_trials(feedback=False)
print("Naive seeded phases:", naive_phases)
assert -1 in naive_phases

A bad sign becomes a relative phase on a superposition. Merely throwing away the measured value does not erase that sign.

## Correcting the phase

The sign is negative only when target=1 and the X outcome=1. If target equals q1 AND q2, a CZ on q1 and q2 conditioned on that outcome cancels it. The native graph uses the existing owned-classical feedback leaf; physical append uses the exact AND adjoint.

In [ ]:
mbuc_target = uncomputation_program(feedback=True)
corrected_circuit = DynamicCircuit(3)
corrected_circuit.append(EraseAnd())
corrected_figure = draw_erasure_steps(corrected_circuit)

## Fuzz testing feedback

All ten corrected runs preserve both controls with final phase one. Sampled successes supplement the exhaustive checks; they do not replace them.

In [ ]:
corrected_phases = seeded_trials(feedback=True)
assert corrected_phases == (1,) * 10
print("Corrected seeded phases:", corrected_phases)

## Exhaustive outcomes

For each of four valid control assignments, force both measurement results. Fixed outcome IDs override RNG through the existing state owner. Additional independent tests cover inconsistent targets and all basis assignments too.

### Finding the measurement occurrence

Compiler call positions supply native persistent occurrence IDs. Here the only measurement has ID zero. No foreign instance objects are required.

In [ ]:
from flagquantum.compiler.registers.component_graph import iter_component_calls
measurement_ids = tuple(i for i, _, _ in iter_component_calls(mbuc_target) if mbuc_target.functions[0].blocks[0].operations[i].callee == "measure_x")
assert measurement_ids == (0,)
measurement_id = measurement_ids[0]

### Inputs and measurement results

Check `sim.phase` for the simulation that just ran. The locked notebook instead asserts a stale variable `phase` from its earlier fuzz loop; changing that variable can fail a correct simulation or hide a bad phase. The reference is unchanged and this difference has an explicit regression.

In [ ]:
exhaustive_rows = []
for q1, q2, outcome in itertools.product(range(2), repeat=3):
    sim = uncomputation_state({"q1": q1, "q2": q2, "target": q1 & q2}, fixed_values={measurement_id: outcome})
    outputs = sim.simulate()
    assert outputs == {"q1": q1, "q2": q2}
    assert sim.phase == 1
    exhaustive_rows.append(((q1, q2, outcome), outputs, sim.phase))
print(exhaustive_rows)

### Inspecting phase after each call

A read-only trace of the current native state replaces the reference debug subclass. The original subclass reads a global `sim` instead of `self`, so it is correct only when the global happens to name that same instance. Native tracing observes its argument directly and avoids a second simulator or global state.

In [ ]:
from examples.algorithms.measurement_uncomputation import phase_trace
# Inspect this state itself. No global sim lookup or stale phase is used.
debug_state = uncomputation_state({"q1": 1, "q2": 1, "target": 1}, fixed_values={measurement_id: 1})
debug_trace = phase_trace(debug_state)
assert debug_trace == (-1, 1, 1)
print("MeasureX / conditional CZ / discard:", debug_trace)

In [ ]:
debug_rows = []
for q1, q2, outcome in itertools.product(range(2), repeat=3):
    state = uncomputation_state({"q1": q1, "q2": q2, "target": q1 & q2}, fixed_values={measurement_id: outcome})
    trace = phase_trace(state)
    assert state.finalize() == {"q1": q1, "q2": q2}
    assert state.phase == 1
    debug_rows.append(((q1, q2, outcome), trace))
print(debug_rows)

Only target=1 and outcome=1 produce a negative intermediate phase. For consistent AND inputs the next conditional CZ immediately cancels it; discard leaves the accumulated phase unchanged. Tests also demonstrate that naive erasure corrupts relative phases on a complex superposition.

Exact physical branch tests retain the projection factor 1/sqrt(2), preserve spectator entanglement, and return target to zero. Primitive gate counts and source QEC models are listed separately in the acceptance document. The source notebook's stale/global-variable defects remain recorded under Q-0879; this asset does not silently repair the reference or close that parent.